In [1]:
# import libaries

import boto3
import pandas as pd
import numpy as np

from io import BytesIO

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [2]:
bucket = 'bucket-name'
key = "ML/delivery_delay_features.csv"

s3 = boto3.client("s3")

response = s3.get_object(
    Bucket = bucket,
    Key = key
)

df_model = pd.read_csv(response["Body"])

In [3]:
df_model.head()

,delayed,scheduled_datetime,dispatch_date,load_type,weight_lbs,pieces,booking_type,origin_state,destination_state,typical_distance_miles,...,truck_brand,model_year,fuel_type,truck_age,dispatch_day_of_week,dispatch_month,dispatch_year,scheduled_hour,scheduled_day_of_week,scheduled_day_of_month
0,0,2022-01-02 23:10:00,2022-01-01,Dry Van,19178,13,Spot,TX,MI,1271,...,International,2015.0,Diesel,7.0,6,1,2022,23,6,2
1,1,2022-01-02 02:13:00,2022-01-01,Dry Van,27761,22,Dedicated,MO,IN,519,...,Volvo,2015.0,Diesel,7.0,6,1,2022,2,6,2
2,0,2022-01-03 04:28:00,2022-01-01,Refrigerated,35594,16,Spot,OH,OR,2332,...,Kenworth,2018.0,Diesel,4.0,6,1,2022,4,0,3
3,1,2022-01-02 06:38:00,2022-01-01,Refrigerated,33274,10,Spot,AZ,CO,673,...,Volvo,2015.0,Diesel,7.0,6,1,2022,6,6,2
4,0,2022-01-02 22:51:00,2022-01-01,Dry Van,40257,10,Spot,TX,WA,2172,...,Kenworth,2015.0,Diesel,7.0,6,1,2022,22,6,2


In [4]:
df_model.shape

(85410, 23)

In [5]:
"""
X = df_model.drop(columns=["delayed"])
Y = df_model["delayed"]
"""

'\nX = df_model.drop(columns=["delayed"])\nY = df_model["delayed"]\n'

In [6]:
df_model = df_model.sort_values("dispatch_date").reset_index(drop=True)

In [7]:
train_end = int(len(df_model) * 0.70)
val_end = int(len(df_model) * 0.85)

train_df = df_model.iloc[:train_end]
val_df = df_model.iloc[train_end:val_end]
test_df = df_model.iloc[val_end:]

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 59786
Validation: 12812
Test: 12812


In [8]:
X_train = train_df.drop(columns=["delayed"])
y_train = train_df["delayed"]

X_val = val_df.drop(columns=["delayed"])
y_val = val_df["delayed"]

X_test = test_df.drop(columns=["delayed"])
y_test = test_df["delayed"]

In [9]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()


print("Categorical:")
print(categorical_features)

print("\nNumerical:")
print(numerical_features)

Categorical:
['scheduled_datetime', 'dispatch_date', 'load_type', 'booking_type', 'origin_state', 'destination_state', 'cdl_class', 'truck_brand', 'fuel_type']

Numerical:
['weight_lbs', 'pieces', 'typical_distance_miles', 'transit_days', 'years_experience', 'model_year', 'truck_age', 'dispatch_day_of_week', 'dispatch_month', 'dispatch_year', 'scheduled_hour', 'scheduled_day_of_week', 'scheduled_day_of_month']


In [10]:
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])


preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [11]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

In [12]:
logistic_model.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:
val_predictions = logistic_model.predict(X_val)

val_probabilities = logistic_model.predict_proba(
    X_val
)[:, 1]

In [14]:
print(
    "Accuracy:",
    accuracy_score(y_val, val_predictions)
)

print(
    "Precision:",
    precision_score(y_val, val_predictions)
)

print(
    "Recall:",
    recall_score(y_val, val_predictions)
)

print(
    "F1:",
    f1_score(y_val, val_predictions)
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_val,
        val_probabilities
    )
)

Accuracy: 0.5341866999687793
Precision: 0.5535048293009467
Recall: 0.8167066459714971
F1: 0.6598267213862289
ROC-AUC: 0.5011741653793803


In [15]:
cm = confusion_matrix(
    y_val,
    val_predictions
)

print(cm)

[[1056 4669]
 [1299 5788]]


### ROC-AUC is 0.501 thats basically random guessing and for the confusion matrix

### True negatives: 1,056
### False positives: 4,669
### False negatives: 1,299
### True positives: 5,788

### there are a lot of false postives

## random forest

In [16]:
rf_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

rf_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

rf_preprocessor = ColumnTransformer([
    ("numeric", rf_numeric_pipeline, numerical_features),
    ("categorical", rf_categorical_pipeline, categorical_features)
])

In [17]:
random_forest = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

In [18]:
random_forest.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('numeric', ...), ('categorical', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
rf_predictions = random_forest.predict(X_val)

rf_probabilities = random_forest.predict_proba(
    X_val
)[:, 1]

In [20]:
rf_accuracy = accuracy_score(
    y_val,
    rf_predictions
)

rf_precision = precision_score(
    y_val,
    rf_predictions
)

rf_recall = recall_score(
    y_val,
    rf_predictions
)

rf_f1 = f1_score(
    y_val,
    rf_predictions
)

rf_roc_auc = roc_auc_score(
    y_val,
    rf_probabilities
)

In [21]:
print("Random Forest Results")
print("---------------------")
print("Accuracy :", rf_accuracy)
print("Precision:", rf_precision)
print("Recall   :", rf_recall)
print("F1 Score :", rf_f1)
print("ROC-AUC  :", rf_roc_auc)

Random Forest Results
---------------------
Accuracy : 0.5438651264439588
Precision: 0.553288176283975
Recall   : 0.910540426132355
F1 Score : 0.68832
ROC-AUC  : 0.4923270666568901


In [22]:
print(
    classification_report(
        y_val,
        rf_predictions
    )
)

              precision    recall  f1-score   support

           0       0.45      0.09      0.15      5725
           1       0.55      0.91      0.69      7087

    accuracy                           0.54     12812
   macro avg       0.50      0.50      0.42     12812
weighted avg       0.51      0.54      0.45     12812



In [23]:
rf_cm = confusion_matrix(
    y_val,
    rf_predictions
)

print(rf_cm)

[[ 515 5210]
 [ 634 6453]]


In [24]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_val, val_predictions),
        rf_accuracy
    ],
    "Precision": [
        precision_score(y_val, val_predictions),
        rf_precision
    ],
    "Recall": [
        recall_score(y_val, val_predictions),
        rf_recall
    ],
    "F1": [
        f1_score(y_val, val_predictions),
        rf_f1
    ],
    "ROC-AUC": [
        roc_auc_score(y_val, val_probabilities),
        rf_roc_auc
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.534187,0.553505,0.816707,0.659827,0.501174
1,Random Forest,0.543865,0.553288,0.910540,0.688320,0.492327


In [25]:
df_model.groupby("delayed").mean(numeric_only=True).T

delayed,0,1
weight_lbs,27519.112829,27443.976431
pieces,14.469687,14.471210
typical_distance_miles,1390.041573,1388.650249
transit_days,2.120361,2.117591
years_experience,13.783184,13.717719
model_year,2015.899159,2015.899983
truck_age,7.101296,7.100578
dispatch_day_of_week,2.997244,2.994377
dispatch_month,6.508713,6.519426
dispatch_year,2022.999974,2023.001437


In [26]:
df_model.corr(numeric_only=True)["delayed"].sort_values()

# results show there is very less to no correlation among the features with the target variable

years_experience         -0.004611
weight_lbs               -0.003700
scheduled_day_of_month   -0.002496
transit_days             -0.001202
typical_distance_miles   -0.000890
scheduled_hour           -0.000729
dispatch_day_of_week     -0.000715
truck_age                -0.000212
pieces                    0.000094
model_year                0.000278
dispatch_year             0.000889
dispatch_month            0.001546
scheduled_day_of_week     0.001575
delayed                   1.000000
Name: delayed, dtype: float64

# Notebook 02 — Delivery Delay Prediction & ML Feasibility Assessment

## Objective

The objective of this notebook was to assess whether the available logistics data could support a reliable machine-learning model for predicting delivery delays.

The modelling process used the feature-engineered `df_model` generated in Notebook 01. The original dataset was not modified during model training.

## Modelling Approach

The data was divided into training, validation and test datasets. Numerical features were imputed and scaled where appropriate, while categorical features were imputed and one-hot encoded.

Two baseline classification models were evaluated:

1. Logistic Regression
2. Random Forest

The models were evaluated using:

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC
- Confusion Matrix

ROC-AUC was treated as the primary indicator of whether the models were able to distinguish between delayed and non-delayed deliveries.

## Logistic Regression Results

| Metric | Result |
|---|---:|
| Accuracy | 53.42% |
| Precision | 55.35% |
| Recall | 81.67% |
| F1 Score | 65.98% |
| ROC-AUC | 0.501 |

Although recall was relatively high, the model generated a large number of false positives. Most importantly, the ROC-AUC of approximately 0.50 indicated that the model had almost no ability to distinguish delayed from non-delayed deliveries beyond random classification.

## Random Forest Results

| Metric | Result |
|---|---:|
| Accuracy | 54.39% |
| Precision | 55.33% |
| Recall | 91.05% |
| F1 Score | 68.83% |
| ROC-AUC | 0.492 |

Random Forest produced a higher recall but did not improve the underlying predictive capability. Its ROC-AUC of approximately 0.49 remained effectively equivalent to random prediction.

## Model Feasibility Assessment

The results were consistent with the findings from the EDA in Notebook 01.

Both linear and tree-based models produced ROC-AUC values close to 0.50. Additional analysis showed that:

- Numerical features had near-zero correlation with the target.
- Engineered operational features provided negligible additional signal.
- Categorical groups showed broadly similar delay proportions.
- Route-level delay rates showed limited variation.

Therefore, the limitation was determined to be primarily **data-related rather than model-related**.

## Decision — ML Development Discontinued

Further model training and hyperparameter tuning were intentionally discontinued.

Rather than artificially increasing model complexity, introducing unsupported assumptions, or overfitting the available data to obtain a higher score, the ML component was treated as a feasibility assessment.

The analysis concluded that the available dataset did not contain sufficient predictive signal to reliably predict delivery delays using the available features.

This is an important data-science outcome: model development should only proceed when the underlying data provides sufficient information to support the prediction objective.

## Final Outcome

The project successfully demonstrated an end-to-end AWS analytics workflow including:

**Amazon S3 → Amazon RDS for PostgreSQL → Data Analysis / Feature Engineering → Amazon SageMaker ML Feasibility Assessment → Grafana Analytics**

The ML assessment provided a data-driven basis for discontinuing predictive modelling rather than deploying an unreliable prediction model.